# Dual-Student Knowledge Distillation (DSKD) - Baseline B0 Training on Kaggle GPU

**CS331 Anomaly Detection Project**  
Reference: *Dual-Student Knowledge Distillation Networks for Unsupervised Anomaly Detection* (arXiv:2402.00448v1)

This notebook trains the faithful baseline `B0-Clean` on Kaggle Tesla T4 GPU.

In [ ]:
# 1. Verify GPU availability
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:     ', torch.cuda.get_device_name(0))
    print(f'VRAM Total:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: No GPU detected. Please go to Kaggle Settings -> Accelerator -> GPU T4!')

In [ ]:
# 2. Clone repository or navigate to source directory
import os
from pathlib import Path

repo_dir = Path('/kaggle/working/cs331-knowledge-distillation')
if not repo_dir.exists():
    !git clone https://github.com/traineethanh/cs331-knowledge-distillation-lightweight-industrial-anomaly-detection.git {repo_dir}

%cd {repo_dir}
!git checkout chore/repository-reset
!pip install -q pyyaml scipy scikit-learn

In [ ]:
# 3. Verify MVTec AD Dataset
mvtec_path = Path('/kaggle/input/mvtec-ad')
if not mvtec_path.exists():
    candidates = list(Path('/kaggle/input').glob('*mvtec*'))
    if candidates:
        mvtec_path = candidates[0]
print(f'Using MVTec AD dataset directory: {mvtec_path}')

categories = [p.name for p in mvtec_path.iterdir() if p.is_dir()]
print(f'Available categories ({len(categories)}): {categories}')

In [ ]:
# 4. Phase 0: Micro-benchmark (M0_bench - 5 epochs on carpet)
# Measures exact wall-clock iteration latency, checkpoint I/O, and CPU AUPRO time
!python scripts/train.py \
    --config configs/baseline_b0.yaml \
    --category carpet \
    --data_root {mvtec_path} \
    --epochs 5 \
    --batch_size 16 \
    --output_dir checkpoints/m0_bench

In [ ]:
# 5. Full Baseline Training (200 epochs on carpet according to paper Section 4.1.2)
# Estimated time on Tesla T4: ~5 minutes!
!python scripts/train.py \
    --config configs/baseline_b0.yaml \
    --category carpet \
    --data_root {mvtec_path} \
    --epochs 200 \
    --batch_size 16 \
    --output_dir checkpoints/baseline_b0

In [ ]:
# 6. Evaluation on Test Set (Image AUROC, Pixel AUROC, Pixel AUPRO@0.30 & 0.05)
!python scripts/evaluate.py \
    --checkpoint checkpoints/baseline_b0/carpet/best_checkpoint.pth \
    --category carpet \
    --data_root {mvtec_path} \
    --output_dir results/baseline_b0/carpet

In [ ]:
# 7. Display Evaluation Metrics Table
import pandas as pd
results_csv = Path('results/baseline_b0/carpet/carpet_eval_metrics.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df)
else:
    print('Results CSV not found yet.')